# Loan Eligibility — SVM with Grid Search

This one specifically uses SVM as the primary model (the brief called it out). Dataset is the well-known "Loan Prediction" practice problem from Analytics Vidhya (Dream Housing Finance), using its labeled `train.csv` (614 applicants) from a public GitHub mirror. Columns are Gender, Married, Dependents, Education, Self_Employed, ApplicantIncome, CoapplicantIncome, LoanAmount, Loan_Amount_Term, Credit_History, Property_Area, and the target Loan_Status.

Why SVM is a reasonable pick here: small dataset (~600 rows), mix of numeric and one-hot encoded categoricals, no need to interpret feature importances. RBF kernel is the obvious starting point. Grid search over C and gamma.

The point of this notebook is the SVM workflow.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score

import urllib.request
np.random.seed(3)
CSV = Path("../../data/loan_eligibility/loan_prediction_train.csv")
URL = "https://raw.githubusercontent.com/shrikant-temburwar/Loan-Prediction-Dataset/master/train.csv"
if not CSV.exists():
    CSV.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(URL, CSV)
df = pd.read_csv(CSV)
print(df.shape)
df.head()


(614, 13)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [2]:
# minimal cleanup: drop rows without Credit_History (50 of 614), fill the other gaps
df = df.dropna(subset=["Credit_History"]).copy()
for c in ["Gender", "Married", "Dependents", "Self_Employed"]:
    df[c] = df[c].fillna(df[c].mode()[0])
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["Loan_Amount_Term"] = df["Loan_Amount_Term"].fillna(360)
print(df.shape, "approval rate:", (df["Loan_Status"] == "Y").mean().round(3))
y = (df["Loan_Status"] == "Y").astype(int).values
X_df = df.drop(columns=["Loan_ID", "Loan_Status"])
X = pd.get_dummies(X_df, drop_first=True).astype(float)
feature_cols = X.columns.tolist()
X = X.values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=3
)
print(f"features: {len(feature_cols)}  train: {len(X_train)}  test: {len(X_test)}")


(564, 13) approval rate: 0.683
features: 14  train: 451  test: 113


## Grid search on C and gamma (RBF kernel)

In [3]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svc",    SVC(kernel="rbf", probability=True, random_state=3)),
])

param_grid = {
    "svc__C":     [0.1, 1, 3, 10, 30],
    "svc__gamma": ["scale", 0.01, 0.05, 0.1, 0.5],
}

# n_jobs helps but the dataset is small enough this runs in seconds anyway
gs = GridSearchCV(pipe, param_grid, cv=StratifiedKFold(5, shuffle=True, random_state=3),
                  scoring="roc_auc", n_jobs=2)
gs.fit(X_train, y_train)
print(f"best params: {gs.best_params_}")
print(f"best CV AUC: {gs.best_score_:.3f}")


best params: {'svc__C': 0.1, 'svc__gamma': 0.1}
best CV AUC: 0.778


In [4]:
best = gs.best_estimator_
proba = best.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
print(f"test AUC: {roc_auc_score(y_test, proba):.3f}")
print(classification_report(y_test, pred, target_names=["denied", "approved"]))


test AUC: 0.804
              precision    recall  f1-score   support

      denied       0.82      0.39      0.53        36
    approved       0.77      0.96      0.86        77

    accuracy                           0.78       113
   macro avg       0.80      0.67      0.69       113
weighted avg       0.79      0.78      0.75       113



## Notes

On the real file (564 applicants after dropping the 50 without a credit history), the grid picks C=0.1 and gamma=0.1, with a best CV AUC of 0.778. On the 113-row test split the SVM gets **AUC 0.804 and 78% accuracy**, but it approves almost everyone: 96% recall on approved loans and only 39% on denied ones.

What jumped out:

- Credit_History dominates everything else. Applicants with a credit history meeting guidelines were approved 80% of the time, versus 8% without. In a side run (not in the cells above) I dropped the column and the SVM's test AUC fell from 0.804 to 0.627.
- The grid lands on a small C, i.e. a heavily regularized boundary. With ~450 training rows and one feature carrying most of the signal, that makes sense.
- The default 0.5 threshold is too lenient for a lender here. Tuning it against the cost of a false approval vs a false denial would be the first real change.

I expected a tree model to edge out the SVM on this categorical-heavy data, but a quick untuned gradient boosting check in the same side run only reached test AUC 0.69, so at least on this split the SVM was the better choice. 113 test rows is small, though, and that comparison could easily flip on a different seed.